# Issue #11: Audience Rating Feature Engineering
Create features for predicting `vote_average`.

## 0. Setup

In [2]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

BASE_PATH = PROJECT_ROOT / "data" / "processed" / "movies_metadata.csv" 

RAW_MOVIES_PATH = PROJECT_ROOT / "data" / "raw" / "movies_metadata.csv"
RAW_CREDITS_PATH = PROJECT_ROOT / "data" / "raw" / "credits.csv"
RAW_KEYWORDS_PATH = PROJECT_ROOT / "data" / "raw" / "keywords.csv"

OUTPUT_PATH = PROJECT_ROOT / "data" / "features" / "07_rating_features.csv"

## 1. Create Helper Functions

In [3]:
def normalize_id(df):
    df = df.copy()
    df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")
    return df.dropna(subset=["id"]).drop_duplicates(subset=["id"])

def parse_list(value):
    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

def format_name_list(names):
    if not names:
        return "[]"

    return "[" + ", ".join(names) + "]"

def format_names(value):
    return format_name_list(get_names(value))

def get_names(value):
    return [
        item.get("name", "")
        for item in parse_list(value)
        if isinstance(item, dict) and item.get("name")
    ]

## 2. Merge Movie Information

In [ ]:
# Load the processed six-column file and raw movie file
base = normalize_id(pd.read_csv(BASE_PATH, low_memory=False))
raw_movies = normalize_id(pd.read_csv(RAW_MOVIES_PATH, low_memory=False))

# Retrieve additional movie columns using id
movie_columns = [
    "id",
    "vote_average",
    "vote_count",
    "genres",
    "belongs_to_collection",
    "production_companies",
    "production_countries",
    "spoken_languages",
    "title",
    "original_title",
    "overview",
    "original_language",
    "status",
]

movie_features = raw_movies[movie_columns]

df = base.merge(
    movie_features,
    on="id",
    how="left",
    validate="one_to_one"
)

## 3. Merge Credits and Keywords Information

In [ ]:
# Retrieve cast and crew information
credits = normalize_id(pd.read_csv(RAW_CREDITS_PATH, low_memory=False))

df = df.merge(
    credits[["id", "cast", "crew"]],
    on="id",
    how="left",
    validate="one_to_one"
)

# Retrieve keyword information
keywords = normalize_id(pd.read_csv(RAW_KEYWORDS_PATH, low_memory=False))

df = df.merge(
    keywords[["id", "keywords"]],
    on="id",
    how="left",
    validate="one_to_one"
)

## 4. Create Date and Numeric Features

In [ ]:
# Date features
dates = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = dates.dt.year.astype("Int64")
df["release_month"] = dates.dt.month.astype("Int64")
df["release_decade"] = (df["release_year"] // 10) * 10

# Numeric features
for column in ["budget", "revenue", "runtime", "popularity", "vote_average"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["vote_count"] = (
    pd.to_numeric(df["vote_count"], errors="coerce")
    .astype("Int64")
)

df["log_budget"] = np.log1p(df["budget"].clip(lower=0))
df["log_revenue"] = np.log1p(df["revenue"].clip(lower=0))
df["log_popularity"] = np.log1p(df["popularity"].clip(lower=0))

## 5. Create List and Credits Features

In [ ]:
# List-based features
df["genre_count"] = df["genres"].apply(lambda x: len(get_names(x)))
df["company_count"] = df["production_companies"].apply(lambda x: len(get_names(x)))
df["country_count"] = df["production_countries"].apply(lambda x: len(get_names(x)))
df["spoken_language_count"] = df["spoken_languages"].apply(lambda x: len(get_names(x)))

df["has_collection"] = df["belongs_to_collection"].notna().astype(int)

# Credits features
df["cast_size"] = df["cast"].apply(lambda x: len(parse_list(x)))
df["crew_size"] = df["crew"].apply(lambda x: len(parse_list(x)))

def get_director(crew):
    for person in parse_list(crew):
        if person.get("job") == "Director":
            return person.get("name")
    return None

df["director"] = df["crew"].apply(get_director)
df["has_director"] = df["director"].notna().astype(int)

df["top_cast"] = df["cast"].apply(
    lambda x: format_name_list(get_names(x)[:5])
)

## 6. Create Keyword and Text Features

In [ ]:
# Keyword and text features
df["keyword_count"] = df["keywords"].apply(lambda x: len(get_names(x)))

df["title_word_count"] = (
    df["title"].fillna("").str.split().str.len()
)

df["overview_word_count"] = (
    df["overview"].fillna("").str.split().str.len()
)

df["title_differs_from_original"] = (
    df["title"].fillna("").str.lower()
    != df["original_title"].fillna("").str.lower()
).astype(int)

## 7. Format List Columns

In [ ]:
for column in [
    "genres",
    "production_companies",
    "production_countries",
    "spoken_languages",
    "keywords",
]:
    df[column] = df[column].apply(format_names)

## 8. Inspect and Filter by Vote Count

In [ ]:
# Inspect vote_count before choosing a threshold
print(df["vote_count"].describe())

# Show how many rows exist before filtering
print(f"Rows before vote_count filtering: {len(df):,}")

# Choose and document your minimum vote count
MIN_VOTE_COUNT = 34
df = df[df["vote_count"] >= MIN_VOTE_COUNT].copy()

count       45430.0
count       45430.0
mean     109.935989
std      491.466335
min             0.0
25%             3.0
50%            10.0
75%            34.0                                            
max         14075.0 
Name: vote_count, dtype: Float64.py
Rows before vote_count filtering: 45,433

## 9. Confirm One Row Per Movie

In [ ]:
assert df["id"].is_unique
print(f"Rows after vote_count filtering: {len(df):,}")
print(f"Unique IDs: {df['id'].nunique():,}")

Rows after vote_count filtering: 11,508
Unique IDs: 11,508

## 10. Select the Final Columns

In [ ]:
final_columns = [
    "id",
    "vote_average",
    "vote_count",
    "release_year",
    "release_month",
    "release_decade",
    "log_budget",
    "log_revenue",
    "runtime",
    "log_popularity",
    "genre_count",
    "has_collection",
    "company_count",
    "country_count",
    "spoken_language_count",
    "cast_size",
    "crew_size",
    "has_director",
    "keyword_count",
    "title_word_count",
    "overview_word_count",
    "title_differs_from_original",
    "genres",
    "production_companies",
    "production_countries",
    "spoken_languages",
    "director",
    "top_cast",
    "keywords",
    "title",
    "original_title",
    "overview",
    "original_language",
    "status",
]

# Create the final dataset using only those columns
final_df = df[final_columns].copy()

# Confirm one row per movie
assert final_df["id"].is_unique

## 11. Save and Verify the Final CSV

In [ ]:
final_df.to_csv(OUTPUT_PATH, index=False)

# Reload and verify the saved file
verified = pd.read_csv(
    OUTPUT_PATH,
    dtype={"vote_count": "Int64"},
    low_memory=False
)

print(f"Saved: {OUTPUT_PATH}")
print(
    f"Final CSV dimensions: "
    f"{verified.shape[0]:,} rows x {verified.shape[1]:,} columns"
)
print("Final columns:")
print(verified.columns.tolist())
print(f"Unique IDs: {verified['id'].nunique():,}")

## Results:
`data\features\07_rating_features.csv`
Final CSV dimensions: 11,508 rows x 34 columns
Final columns:
['id', 'vote_average', 'vote_count', 'release_year', 'release_month', 'release_decade', 'log_budget', 'log_revenue', 'runtime', 'log_popularity', 'genre_count', 'has_collection', 'company_count', 'country_count', 'spoken_language_count', 'cast_size', 'crew_size', 'has_director', 'keyword_count', 'title_word_count', 'overview_word_count', 'title_differs_from_original', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'director', 'top_cast', 'keywords', 'title', 'original_title', 'overview', 'original_language', 'status']
Unique IDs: 11,508